In [1]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os

In [2]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION'

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
from Unet_model import UNet

In [5]:
my_model = UNet()
my_model.load_state_dict(torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/unet/unet_rooftop_50_2.pth"))
my_model.to(device=device)

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [7]:
root_dir = '/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/val'

val_images_dir = os.path.join(root_dir, 'images')
val_masks_dir = os.path.join(root_dir, 'masks')

In [8]:
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms
import cv2

In [9]:
# Custom Dataset
class RooftopDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_filenames = sorted(os.listdir(image_dir))
        self.transform = transform
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir, self.image_filenames[idx])
        
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = mask.astype(np.float32) / 255.0
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask).unsqueeze(0)  # Ensure mask shape [1, H, W]
        
        return image, mask.squeeze(0)  # Ensure mask shape [H, W]

# Transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((1024, 1024)),
    transforms.ToTensor()
])

# Load datasets
val_dataset = RooftopDataset(val_images_dir, val_masks_dir, transform)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)


In [10]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION'

In [11]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection'

In [12]:
from Solar_Rooftop_Detection.accuracy import compute_metrics

In [13]:
# Evaluation

def evaluate(model, val_loader):
    model.eval()
    result_data = []
    # iou_scores = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            # print(outputs.shape)
            preds = (outputs > 0.5).float()
            
            # one  = preds[0].squeeze(0).cpu().numpy()
            # plt.imshow(one, cmap="gray")
            # plt.tight_layout()
            # plt.show()
            
            # original = masks[0].squeeze(0).cpu().numpy()
            # plt.imshow(original, cmap="gray")
            # plt.tight_layout()
            # plt.show()

            metrics = compute_metrics(preds, masks)
            result_data.append(metrics)

    return result_data

In [14]:
accuracy_result = evaluate(my_model, val_loader)
accuracy_result

[[np.float64(0.8492281709711796),
  0.9184676983643193,
  0.9461394548416138,
  0.8954432898460337,
  0.9427074024223119,
  0.8492281709711796,
  0.9184676983643193,
  0.8954432898460337,
  0.9427074024223119,
  1.0],
 [np.float64(0.6776668842965954),
  0.8078682253786329,
  0.9166040420532227,
  0.7050845523789937,
  0.945732316810279,
  0.6776668842965954,
  0.8078682253786329,
  0.7050845523789937,
  0.945732316810279,
  1.0],
 [np.float64(0.8958214561130913),
  0.9450483358804785,
  0.9810372591018677,
  0.97190620836664,
  0.919634939278686,
  0.8958214561130913,
  0.9450483358804785,
  0.97190620836664,
  0.919634939278686,
  1.0],
 [np.float64(0.8732490451999816),
  0.9323363035337958,
  0.9754927158355713,
  0.9531190855038676,
  0.912440518582296,
  0.8732490451999816,
  0.9323363035337958,
  0.9531190855038676,
  0.912440518582296,
  1.0],
 [np.float64(0.8973187877811385),
  0.9458808857635651,
  0.9811592102050781,
  0.9351621580813624,
  0.9568481762444144,
  0.897318787781

In [15]:
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/unet/unet_validation_results_1.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

In [16]:
import pandas as pd 

metrics_df = pd.DataFrame(accuracy_result, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

In [17]:
metrics_df.mean()

pixel_iou                  0.914100
pixel_dice                 0.953941
pixel_accuracy             0.976602
pixel_precision            0.953390
pixel_recall               0.956509
region_iou                 0.914100
region_dice                0.953941
region_precision           0.953390
region_recall              0.956509
region_success_accuracy    1.000000
dtype: float64